# פרק 16 — מעבדת תלמיד: תכנון עבודת גמר בלמידה עמוקה
**מסלול התמחות 5 יח״ל | 12 משימות עצמאיות**

המטרה: להתנסות בתהליך מסודר של בניית מערכת AI על נתוני `digits`. אין כאן פתרון מלא לפרויקט גמר. יש להגיש תיעוד מקורי ולהתאים את הפרויקט לדרישות הפיקוח התקפות.

## פתיחה — סביבה ונתונים
הריצו את תאי ההכנה. `X` מכיל 1,797 תמונות שטוחות של 64 פיקסלים; `y` מכיל את התוויות. **אין לגעת בקבוצת Test לצורך בחירת המודל.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
import torch
from torch import nn

SEED = 23
np.random.seed(SEED)
torch.manual_seed(SEED)
digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(np.int64)
X_train_all, X_test, y_train_all, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all, test_size=0.25,
    stratify=y_train_all, random_state=SEED
)
print('shapes:', X_train.shape, X_val.shape, X_test.shape)


### משימה 1 — כרטיס פרויקט
נסחו משתמש, צורך, מטרת מודל ומדד הצלחה. כתבו מדוע לא מספיק להפעיל מסווג במחברת בלבד.

In [ ]:
# TODO: כתבו בתא Markdown חדש את כרטיס הפרויקט שלכם.


### משימה 2 — כרטיס נתונים
בדקו מקור המאגר, שדות, מחלקות, תחום ערכים, זכויות שימוש ומגבלות. הסבירו מדוע ציור חדש מחוץ למאגר עלול להיות שונה מתמונות האימון.

In [ ]:
print('dataset:', digits.DESCR.splitlines()[0])
print('pixel range:', X_train.min(), X_train.max())
# TODO: בדקו שכיחות של עשר המחלקות וכתבו מסקנה.


### משימה 3 — חקר נתונים
הציגו ארבע תמונות, היסטוגרמה או גרף עמודות של התוויות, ומסקנה שאינה חורגת מן הנתונים.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
for idx, ax in enumerate(axes):
    ax.imshow(X_train[idx].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'Label {y_train[idx]}')
    ax.axis('off')
plt.show()
# TODO: הוסיפו גרף של התפלגות התוויות והסבירו אותו.


### משימה 4 — Baseline
אמנו מסווג פשוט שמנבא את המחלקה השכיחה, ובדקו את תוצאתו על Validation. מה הוא מלמד אותנו על משמעות המדד?

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
print('Validation baseline:', accuracy_score(y_val, baseline.predict(X_val)))
# TODO: כתבו פרשנות מדוע baseline זה חלש בבעיה זו.


### משימה 5 — תכנון רשת
בנו תרשים קלט → שכבה חבויה → פלט עבור 64–32–10. חשבו ידנית את מספר הפרמטרים, והסבירו מהו logit.

In [ ]:
class DigitMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10)
        )
    def forward(self, x):
        return self.net(x)

model = DigitMLP()
print('Parameters:', sum(p.numel() for p in model.parameters()))
# TODO: הוכיחו את הספירה בחישוב ידני.


### משימה 6 — קדם־עיבוד ללא זליגה
הסבירו מדוע החלוקה ב־16 מתאימה **לנתונים האלה** ולא בהכרח לתמונות רגילות. רשמו מה יהיה תהליך קדם־עיבוד ביישום.

In [ ]:
X_train_n = torch.tensor(X_train / 16.0)
X_val_n = torch.tensor(X_val / 16.0)
y_train_t = torch.tensor(y_train)
y_val_t = torch.tensor(y_val)
# TODO: בדקו shape וטווחים של המערכים המנורמלים.


### משימה 7 — ניסוי אימון ראשון
השלימו את לולאת האימון. בצעו חיזוי ומדידה על Validation. שמרו את תוצאות הניסוי ביומן; אין להשתמש ב־Test לבחירת הארכיטקטורה.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
# TODO: כתבו לולאת Epochs. בכל Epoch בצעו:
# model.train(); optimizer.zero_grad(); loss.backward(); optimizer.step()
# וחשבו Accuracy על Validation ב-model.eval() עם torch.no_grad().


### משימה 8 — יומן ניסויים
השוו שתי הגדרות של קצב למידה או רוחב שכבה. מה היתה ההשערה? מה נמדד? האם השינוי הסביר שיפור יציב?

In [ ]:
experiments = pd.DataFrame(columns=['experiment','change','val_accuracy','interpretation'])
display(experiments)
# TODO: הוסיפו שתי רשומות לאחר הפעלת המודלים.


### משימה 9 — בדיקה סופית
לאחר שהשלמתם את בחירת המודל, הריצו Test פעם אחת ודווחו Accuracy ו־Macro F1, לצד מטריצת בלבול וטעויות מייצגות. הסבירו מה לא ניתן להסיק מן המדד.

In [ ]:
# TODO: חשבו X_test_n = torch.tensor(X_test/16.0)
# TODO: הריצו את המודל שנבחר ושמרו תחזיות.
# TODO: חשבו confusion_matrix ו-classification_report.


### משימה 10 — חוזה לקלט יישום
תכננו פונקציה שמקבלת תמונה 8×8, בודקת טווח 0–16 ומחזירה וקטור קלט מנורמל 1×64. כתבו שתי בדיקות לקלט שגוי.

In [ ]:
def prepare_digit(image):
    # TODO: בדקו גודל, ערכים מספריים וטווח; החזירו (1,64)
    raise NotImplementedError


### משימה 11 — ארכיטקטורת המערכת
ציירו תרשים UI → Input Validation → Preprocessing → Model → Display. תארו כיצד המודל נשמר ונטען ובמה יבדקו שהיישום עובד גם בלי מחברת האימון.

In [ ]:
# TODO: הוסיפו תא Markdown עם תרשים מערכת ומקרי בדיקה.


### משימה 12 — הכנה להגנה ולתיק הפרויקט
כתבו חמישה פריטים: מגבלה, החלטה מקצועית, ניסוי שנכשל, שינוי עתידי אפשרי ושאלה שהייתם מצפים לה מן הבוחן. צרפו קישור לקוד ולדוח בהתאם להנחיות בית הספר.

In [ ]:
# TODO: מסמך סיכום קצר, קישורים להוכחות, יומן ניסויים ורפלקציה.


---
**כללי הגשה:** אין לפרסם נתונים אישיים; יש לציין מקורות ורישיונות. משימות אלה הן תרגול של תהליך עבודה ואינן מחליפות את החובה ליצור מערכת עצמאית הכוללת נתונים, מודל עמוק ויישום תוכנה, בהתאם למחוון התקף.